In [ ]:
!pip install GEOparse --quiet

!pip install pyensembl -- quiet

!pip install joblib scikit-learn seaborn matplotlib --quiet

# =============================================================================
# External Validation of MS Progression Model (RRMS vs. SPMS)
# Training Data : GSE17048  (whole blood, Microarray)
# Validation Data: GSE247181 (whole blood, RNA-Seq) ← corrected dataset
# =============================================================================



# ── CELL 1: Environment Setup ─────────────────────────────────────────────────
# !pip install GEOparse joblib --quiet

import warnings, os, re, pickle, joblib
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import GEOparse

from itertools import combinations
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)

print("✅ All libraries loaded.")


# ── CELL 2: Configuration ─────────────────────────────────────────────────────
# Replace with your actual gene_importance.head(10)['Gene'].tolist() output.
TOP_10_GENES = [
    'AQP9', 'RPS11', 'RPS12', 'RPS17',
    'TYROBP', 'LYL1', 'EIF3E', 'SNX27',
    'TMEM123', 'LOC441087'
]

# Ensembl-to-symbol map. GSE247181 may already use gene symbols; this is a
# safety net for datasets that store Ensembl IDs.
ENSEMBL_TO_SYMBOL = {
    'ENSG00000103569': 'AQP9',
    'ENSG00000142178': 'RPS11',
    'ENSG00000197756': 'RPS12',
    'ENSG00000105372': 'RPS17',
    'ENSG00000011600': 'TYROBP',
    'ENSG00000104903': 'LYL1',
    'ENSG00000070061': 'EIF3E',
    'ENSG00000113070': 'SNX27',
    'ENSG00000160218': 'TMEM123',
}

GEO_ID     = "GSE247181"   # whole-blood RNA-seq, RRMS vs SPMS
MODEL_PATH = "model.pkl"
THRESHOLD  = 0.40

print(f"✅ Config set. Target dataset: {GEO_ID}")


# ── CELL 3: Download GSE247181 ────────────────────────────────────────────────
print(f"\n⬇️  Downloading {GEO_ID} from NCBI GEO…")
gse = GEOparse.get_GEO(geo=GEO_ID, destdir="./geo_data", silent=True)
print(f"✅ Title: {gse.metadata['title'][0]}")
print(f"   GSMs : {len(gse.gsms)}")

# ── DIAGNOSTIC: print first sample's metadata so you can see label fields ─────
first_gsm = next(iter(gse.gsms.values()))
print("\n── First GSM metadata (check for label fields) ──")
for k, v in first_gsm.metadata.items():
    print(f"  {k}: {v}")


# ── CELL 4: Metadata Extraction — adaptive label parsing ─────────────────────
# GSE247181 stores diagnosis in 'characteristics_ch1'. The exact key name
# varies by dataset, so we search all text fields for RRMS / SPMS tokens.

records = []
for gsm_name, gsm in gse.gsms.items():
    meta    = gsm.metadata
    title   = meta.get('title', [''])[0]
    source  = meta.get('source_name_ch1', [''])[0]
    chars   = '; '.join(meta.get('characteristics_ch1', []))
    desc    = meta.get('description', [''])[0]
    combined = f"{title} {source} {chars} {desc}".upper()

    if   'SPMS' in combined or 'SECONDARY PROGRESSIVE' in combined:
        label = 'SPMS'
    elif 'RRMS' in combined or 'RELAPSING' in combined:
        label = 'RRMS'
    elif 'HEALTHY' in combined or 'CONTROL' in combined or 'HC' in combined:
        label = 'HC'
    else:
        label = None

    records.append({'GSM': gsm_name, 'title': title,
                    'source': source, 'characteristics': chars, 'label': label})

meta_df = pd.DataFrame(records)

# Keep only RRMS and SPMS (drop HC and unknowns)
valid_df = meta_df[meta_df['label'].isin(['RRMS', 'SPMS'])].copy().reset_index(drop=True)

print(f"\n📊 Label distribution (RRMS vs SPMS only):")
print(valid_df['label'].value_counts().to_string())

if valid_df.empty:
    print("\n⚠️  No RRMS/SPMS samples found via keyword search.")
    print("    Run the DIAGNOSTIC block above and identify the correct field.")
    print("    Then update the label-parsing logic in Cell 4 accordingly.")
    raise SystemExit("Stopping here — update label parsing before continuing.")

print(f"\nSample preview:")
print(valid_df[['GSM', 'title', 'label']].head(10).to_string(index=False))


# ── CELL 5: Expression Data Acquisition ──────────────────────────────────────
# GSE247181 is RNA-seq submitted via GEO soft format. We try three strategies:
#   A) GEOparse pivot (works if VALUE column is present)
#   B) Supplemental file download from NCBI FTP
#   C) E-MTAB-13378 ArrayExpress mirror (same dataset, different host)

gsm_keep = valid_df['GSM'].tolist()
expr_raw  = None

# --- Strategy A: GEOparse pivot -----------------------------------------------
print("\n📂 Strategy A: GEOparse pivot…")
try:
    pivot = gse.pivot_samples('VALUE')
    overlap = [c for c in pivot.columns if c in gsm_keep]
    if len(overlap) == 0:
        raise ValueError("No matching GSMs in pivot.")
    expr_raw = pivot[overlap]
    print(f"✅ Pivot OK. Shape: {expr_raw.shape}")
except Exception as e:
    print(f"⚠️  Pivot failed: {e}")

# --- Strategy B: Supplemental FTP file ----------------------------------------
if expr_raw is None:
    print("\n📂 Strategy B: Supplemental file download…")
    # Check GEO supplemental files listed in the series metadata
    suppl_files = gse.metadata.get('supplementary_file', [])
    print(f"   Supplemental files listed: {suppl_files}")

    # Common filename patterns for this study
    candidates = [
        "GSE247181_raw_counts.txt.gz",
        "GSE247181_counts.csv.gz",
        "GSE247181_fpkm.txt.gz",
    ]
    # Add any files listed in metadata
    for url in suppl_files:
        fname = url.split('/')[-1]
        candidates.insert(0, (url, fname))

    for item in candidates:
        if isinstance(item, tuple):
            url, fname = item
        else:
            url  = f"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE247nnn/GSE247181/suppl/{item}"
            fname = item

        if not os.path.exists(fname):
            ret = os.system(f"wget -q -O {fname} '{url}'")

        if os.path.exists(fname) and os.path.getsize(fname) > 5000:
            try:
                compression = 'gzip' if fname.endswith('.gz') else None
                sep = '\t' if '.txt' in fname else ','
                df_tmp = pd.read_csv(fname, sep=sep, index_col=0, compression=compression)
                col_match = [c for c in df_tmp.columns if any(g in c for g in gsm_keep)]
                if col_match:
                    expr_raw = df_tmp[col_match]
                else:
                    expr_raw = df_tmp   # columns may be sample names, not GSM IDs
                print(f"✅ Loaded supplemental: {fname}  shape={expr_raw.shape}")
                break
            except Exception as e:
                print(f"   Failed to parse {fname}: {e}")
                os.remove(fname)

# --- Strategy C: ArrayExpress mirror (E-MTAB-13378) ---------------------------
# The OmicsDI entry for GSE247181 cross-references E-MTAB-13378.
if expr_raw is None:
    print("\n📂 Strategy C: ArrayExpress E-MTAB-13378…")
    ae_url  = "https://ftp.ebi.ac.uk/biostudies/fire/E-MTAB-/378/E-MTAB-13378/Files/raw_counts.txt"
    ae_file = "E-MTAB-13378_counts.txt"
    if not os.path.exists(ae_file):
        os.system(f"wget -q -O {ae_file} '{ae_url}'")
    if os.path.exists(ae_file) and os.path.getsize(ae_file) > 5000:
        try:
            expr_raw = pd.read_csv(ae_file, sep='\t', index_col=0)
            print(f"✅ ArrayExpress file loaded. Shape: {expr_raw.shape}")
        except Exception as e:
            print(f"   Failed: {e}")

# --- Fallback: clearly-labelled synthetic data --------------------------------
if expr_raw is None:
    print("\n⚠️  All download strategies failed.")
    print("    ACTION REQUIRED: manually download the counts matrix from:")
    print("    https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE247181")
    print("    Save it to the working directory, then re-run Cell 5.")
    print("\n    Creating a labelled synthetic matrix for PIPELINE TESTING only.")
    print("    Results from synthetic data are NOT biologically meaningful.\n")

    np.random.seed(42)
    n_genes   = 2000
    n_samples = len(gsm_keep) if gsm_keep else 13  # GSE247181 has ~13 samples

    # Inject target gene names so downstream cells work
    gene_ids  = [f"GENE_{i}" for i in range(n_genes)]
    for i, sym in enumerate(TOP_10_GENES):
        if i < n_genes:
            gene_ids[i] = sym

    if not gsm_keep:
        # Create dummy GSM IDs matching the metadata length (n_rrms=5, n_spms=8)
        gsm_keep = [f"GSM_RRMS_{i}" for i in range(5)] + \
                   [f"GSM_SPMS_{i}" for i in range(8)]
        valid_df = pd.DataFrame({
            'GSM'  : gsm_keep,
            'label': ['RRMS']*5 + ['SPMS']*8
        })

    expr_raw = pd.DataFrame(
        np.random.negative_binomial(n=10, p=0.5, size=(n_genes, len(gsm_keep))),
        index   = gene_ids,
        columns = gsm_keep
    ).astype(float)
    print(f"   Synthetic data shape: {expr_raw.shape}")


# ── CELL 6: Align columns to valid samples ────────────────────────────────────
# expr_raw columns may be GSM IDs, sample names, or numeric indices.
# We try to match them to valid_df['GSM'] by substring.
col_map = {}
for col in expr_raw.columns:
    for gsm in gsm_keep:
        if gsm in str(col) or str(col) in gsm:
            col_map[col] = gsm
            break

if col_map:
    expr_raw = expr_raw[list(col_map.keys())].rename(columns=col_map)
    print(f"✅ Column alignment: {len(col_map)} samples matched.")
else:
    # Columns are already GSM IDs or already aligned
    matched = [c for c in expr_raw.columns if c in gsm_keep]
    if matched:
        expr_raw = expr_raw[matched]
    else:
        print("⚠️  Could not match expression columns to GSM IDs.")
        print(f"   Expression columns (first 5): {list(expr_raw.columns[:5])}")
        print(f"   Expected GSMs (first 5): {gsm_keep[:5]}")
        print("   Proceeding with unmatched columns (labels may be misaligned).")
        # Use column order to assign labels if counts match
        if expr_raw.shape[1] == len(gsm_keep):
            expr_raw.columns = gsm_keep

gsm_cols  = [c for c in gsm_keep if c in expr_raw.columns]
valid_df  = valid_df[valid_df['GSM'].isin(gsm_cols)].reset_index(drop=True)
expr_raw  = expr_raw[gsm_cols]
print(f"   Final expression matrix: {expr_raw.shape}")


# ── CELL 7: Gene ID Mapping (Ensembl → Symbol) ───────────────────────────────
print("\n🔁 Mapping gene identifiers…")

def strip_version(x):
    return str(x).split('.')[0]

expr_raw.index = expr_raw.index.map(strip_version)
expr_raw.index = expr_raw.index.map(lambda x: ENSEMBL_TO_SYMBOL.get(x, x))
expr_mapped    = expr_raw.groupby(expr_raw.index).median()

present_genes = [g for g in TOP_10_GENES if g in expr_mapped.index]
missing_genes = [g for g in TOP_10_GENES if g not in expr_mapped.index]

print(f"✅ Genes found    ({len(present_genes)}): {present_genes}")
if missing_genes:
    print(f"⚠️  Genes missing ({len(missing_genes)}): {missing_genes}")
    print("   Missing genes excluded from ratio pairs gracefully.")

if len(present_genes) < 2:
    raise ValueError(
        "Fewer than 2 target genes found in the expression matrix. "
        "Check your gene mapping or the downloaded file format."
    )


# ── CELL 8: Log₂ Ratio Feature Engineering ───────────────────────────────────
print("\n⚙️  Building log₂ ratio features…")

X_genes = expr_mapped.loc[present_genes, gsm_cols].T
X_genes = X_genes.apply(pd.to_numeric, errors='coerce').fillna(X_genes.median())

# CPM normalisation for raw counts (safe to apply to FPKM/TPM too)
col_sums = X_genes.sum(axis=1)
col_sums[col_sums == 0] = 1
X_cpm  = X_genes.div(col_sums, axis=0) * 1e6
X_log  = np.log2(X_cpm + 1)

ratio_dict = {}
for g1, g2 in combinations(present_genes, 2):
    ratio_dict[f"{g1}_{g2}_logratio"] = (X_log[g1] - X_log[g2]).values

ratios_df = pd.DataFrame(ratio_dict, index=gsm_cols)
ratios_df.replace([np.inf, -np.inf], np.nan, inplace=True)
ratios_df.fillna(ratios_df.median(), inplace=True)

label_map = valid_df.set_index('GSM')['label'].to_dict()
y_labels  = np.array([label_map[s] for s in gsm_cols])
y_binary  = (y_labels == 'SPMS').astype(int)   # SPMS=1, RRMS=0

print(f"✅ Feature matrix : {ratios_df.shape}  (samples × ratio features)")
print(f"   Labels         : RRMS={int((y_binary==0).sum())}, SPMS={int((y_binary==1).sum())}")


# ── CELL 9: ThresholdClassifier definition ───────────────────────────────────
class ThresholdClassifier(ClassifierMixin, BaseEstimator):
    """Sklearn-compatible wrapper that applies a custom probability threshold."""
    def __init__(self, estimator, threshold=0.40):
        self.estimator = estimator
        self.threshold = threshold
        self.classes_  = np.array([0, 1])

    def fit(self, X, y):
        self.estimator.fit(X, y)
        self.classes_ = np.array([0, 1])
        return self

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)


# ── CELL 10: Model Loading or Fallback Re-training ───────────────────────────
if os.path.exists(MODEL_PATH):
    print(f"\n✅ Loading pre-trained model from '{MODEL_PATH}'…")
    model = joblib.load(MODEL_PATH)
    print("   Model loaded. Using for true external validation.")
else:
    print(f"\n⚠️  '{MODEL_PATH}' not found.")
    print("   Training a surrogate SVM on this validation cohort for pipeline testing.")
    print("   For real external validation, upload the model trained on GSE17048.\n")

    if len(gsm_cols) < 4:
        raise ValueError(
            f"Only {len(gsm_cols)} samples available — too few to train/test. "
            "Supply the pre-trained model or obtain more samples."
        )

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svc',    SVC(kernel='linear', C=0.1, probability=True,
                       class_weight='balanced', max_iter=20000))
    ])
    model = ThresholdClassifier(pipe, threshold=THRESHOLD)
    model.fit(ratios_df, y_binary)
    print("✅ Surrogate model trained.")


# ── CELL 11: Feature Alignment & Prediction ──────────────────────────────────
print("\n🔍 Running predictions…")

if hasattr(model, 'feature_names_in_'):
    # Zero-fill features the model expects but validation lacks
    missing_ft = [c for c in model.feature_names_in_ if c not in ratios_df.columns]
    for col in missing_ft:
        ratios_df[col] = 0.0
    extra_ft = [c for c in ratios_df.columns if c not in model.feature_names_in_]
    X_val = ratios_df[model.feature_names_in_]
    if missing_ft:
        print(f"   ⚠️  Zero-filled {len(missing_ft)} missing model features.")
    if extra_ft:
        print(f"   ℹ️  Dropped {len(extra_ft)} extra features not seen at training.")
else:
    X_val = ratios_df

y_probs = model.predict_proba(X_val)[:, 1]
y_pred  = (y_probs >= THRESHOLD).astype(int)

print("\n── Classification Report ──────────────────────────────────────────────")
print(classification_report(y_binary, y_pred, target_names=['RRMS', 'SPMS'],
                             zero_division=0))


# ── CELL 12: Visualisation — Confusion Matrix + ROC ──────────────────────────
fpr, tpr, _ = roc_curve(y_binary, y_probs, drop_intermediate=False)
roc_auc     = auc(fpr, tpr)
cm          = confusion_matrix(y_binary, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"External Validation — {GEO_ID} (Whole-Blood RNA-Seq)\nRRMS vs. SPMS",
    fontsize=14, fontweight='bold', y=1.02
)

# Confusion matrix
ConfusionMatrixDisplay(cm, display_labels=['RRMS', 'SPMS']).plot(
    ax=axes[0], cmap='Blues', colorbar=False
)
axes[0].set_title('Confusion Matrix', fontsize=12, pad=12)

# ROC curve
axes[1].plot(fpr, tpr, color='royalblue', lw=3, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC = 0.500)')
axes[1].fill_between(fpr, tpr, alpha=0.10, color='royalblue')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate',  fontsize=11)
axes[1].set_title('ROC Curve — External Validation', fontsize=12, pad=12)
axes[1].legend(loc='lower right', fontsize=11)
axes[1].grid(alpha=0.3)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('validation_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved → validation_performance.png")


# ── CELL 13: Heatmap of Top Genes ────────────────────────────────────────────
if len(present_genes) >= 2:
    palette    = {'RRMS': '#4E9AF1', 'SPMS': '#E05C5C'}
    row_colors = pd.Series(y_labels, index=gsm_cols, name='MS Type').map(palette)

    g = sns.clustermap(
        X_log[present_genes].T,
        cmap             = 'vlag',
        standard_scale   = 0,
        col_colors       = row_colors,
        figsize          = (13, max(5, len(present_genes) * 0.7 + 2)),
        method           = 'ward',
        metric           = 'euclidean',
        linewidths       = 0.3,
        yticklabels      = True,
        dendrogram_ratio = (0.15, 0.10),
        cbar_pos         = (0.02, 0.83, 0.03, 0.15)
    )

    from matplotlib.patches import Patch
    handles = [Patch(color=c, label=l) for l, c in palette.items()]
    g.ax_heatmap.legend(handles=handles, title='MS Type',
                        loc='upper right', bbox_to_anchor=(1.18, 1.15),
                        frameon=True, fontsize=10)
    g.fig.suptitle(
        f'Top {len(present_genes)} Genes: RRMS vs. SPMS ({GEO_ID}, log₂ CPM)',
        y=1.01, fontsize=13, fontweight='bold'
    )
    plt.savefig('top_genes_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved → top_genes_heatmap.png")
else:
    print("⚠️  Fewer than 2 genes present — skipping heatmap.")


# ── CELL 14: Summary ──────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  EXTERNAL VALIDATION SUMMARY")
print("="*60)
print(f"  Validation dataset : {GEO_ID} (whole blood RNA-Seq)")
print(f"  Samples evaluated  : {len(gsm_cols)}  "
      f"(RRMS={int((y_binary==0).sum())}, SPMS={int((y_binary==1).sum())})")
print(f"  Target genes found : {len(present_genes)} / {len(TOP_10_GENES)}")
print(f"  Missing genes      : {missing_genes if missing_genes else 'None'}")
print(f"  Ratio features     : {ratios_df.shape[1]}")
print(f"  Decision threshold : {THRESHOLD}")
print(f"  AUC-ROC            : {roc_auc:.3f}")
print("="*60)
print("\n  Output files:")
print("    • validation_performance.png  (confusion matrix + ROC)")
print("    • top_genes_heatmap.png       (clustermap)")
print("="*60)